# Practice 2 — Sweep fine-tune trên GPU Colab

Sweep 2 siêu tham số đúng yêu cầu đề bài:
- learning rate: 5 giá trị {0.0001, 0.0005, 0.001, 0.003, 0.005}
- batch-size: {32, 64}
- 3 mô hình: resnet18, vgg11, densenet121 — strategy fine_tune, optimizer adam
- Tổng 3 x 5 x 2 = 30 cấu hình, mỗi cái 5 epoch, ảnh 64.

Chạy từng ô theo thứ tự từ trên xuống dưới.

## 1. Kiểm tra GPU (phải hiện T4 mới đúng)

In [1]:
import torch
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

Tesla T4


## 2. Lấy code từ GitHub

In [2]:
REPO_URL = "https://github.com/Sper-il/Lab2.git"

import os
if not os.path.isdir("/content/Lab2"):
    !git clone $REPO_URL /content/Lab2
else:
    !git -C /content/Lab2 pull
%cd /content/Lab2
!git log --oneline -3
!git status --short

Cloning into '/content/Lab2'...
remote: Enumerating objects: 434, done.
remote: Counting objects: 100% (434/434), done.
remote: Compressing objects: 100% (261/261), done.
remote: Total 434 (delta 205), reused 370 (delta 144), pack-reused 0 (from 0)
Receiving objects: 100% (434/434), 4.07 MiB | 32.08 MiB/s, done.
Resolving deltas: 100% (205/205), done.
/content/Lab2
fb8cfd5 (HEAD -> main, origin/main, origin/HEAD) chore: chinh sweep Colab ve 5 epoch
9c5037b perf: bat AMP + non_blocking + workers=2 cho GPU T4
b5183f8 perf: toi uu DataLoader cho GPU T4 (pin_memory, persistent workers)


## 3. Cài thư viện

In [3]:
!pip -q install tensorboard pyyaml matplotlib pillow numpy
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

torch 2.11.0+cu130 | torchvision 0.26.0+cu130


## 4. Tải CIFAR-10 về máy Colab

In [4]:
!python scripts/download_data.py
!python scripts/verify_data.py

INFO | Đang tải CIFAR-10 vào /content/Lab2/data/raw
100% 170M/170M [00:16<00:00, 10.5MB/s]
INFO | Tải xong trong 20.7 giây
INFO | Train: 50000 mẫu, 10 lớp
INFO | Test:  10000 mẫu
INFO | Các lớp: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
INFO | Archive giữ lại tại /content/Lab2/data/raw/cifar-10-python.tar.gz (162.6 MB)
INFO | MD5 archive OK: c58f30108f718f92721af3b95e74349a (162.6 MB)
INFO | Train: 50000 mẫu | Test: 10000 mẫu
INFO | Shape ảnh đầu tiên: (32, 32) | label: 6
INFO | Phân bố lớp train: {0: 5000, 1: 5000, 2: 5000, 3: 5000, 4: 5000, 5: 5000, 6: 5000, 7: 5000, 8: 5000, 9: 5000}
INFO | Phân bố lớp test:  {0: 1000, 1: 1000, 2: 1000, 3: 1000, 4: 1000, 5: 1000, 6: 1000, 7: 1000, 8: 1000, 9: 1000}
INFO | Số lớp: 10 | cân bằng: True
INFO | Tất cả kiểm tra PASS — dữ liệu sẵn sàng dùng.


## 5. Chạy nhanh 1 cấu hình thử máy (1 epoch)

In [5]:
!python -m src.training --config configs/train_config.yaml --models resnet18 --lr 0.001 --batch-size 32 --epochs 1 --image-size 64 --strategy fine_tune --optimizer adam --no-tensorboard

I0000 00:00:1793374928.887509    2579 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793374932.906325    2579 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Run parent: /content/Lab2/results/runs/20261030-154216 | so cau hinh: 1
[1/1] resnet18-lr0.001-bs32-0
INFO    | Split train/val: 45000 train / 5000 val (seed=42, 10 lớp)
INFO    | DataLoader: train=1407 batch | val=157 batch | test=313 batch | workers=2
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 198MB/s]
INFO    | epoch 1/1 | train loss 0.8843 acc 0.6988 | val loss 0.6006 acc 0.7868 | lr 1.00e-03 | 37.6s *best*
Sweep summary: /content/Lab2/results/

## 6. Chạy sweep 30 cấu hình (lâu nhất — cứ để chạy)

In [6]:
!python -m src.training --config configs/train_config.yaml --models resnet18 vgg11 densenet121 --lr 0.0001 0.0005 0.001 0.003 0.005 --batch-size 32 64 --epochs 5 --image-size 64 --strategy fine_tune --optimizer adam

I0000 00:00:1793374982.214335    2876 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793374984.440711    2876 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Run parent: /content/Lab2/results/runs/20261030-154307 | so cau hinh: 30
[1/30] resnet18-lr0.0001-bs32-0
INFO    | Split train/val: 45000 train / 5000 val (seed=42, 10 lớp)
INFO    | DataLoader: train=1407 batch | val=157 batch | test=313 batch | workers=2
INFO    | epoch 1/5 | train loss 0.9063 acc 0.6852 | val loss 0.6489 acc 0.7742 | lr 1.00e-04 | 33.9s *best*
INFO    | epoch 2/5 | train loss 0.6633 acc 0.7674 | val loss 0.5377 acc 0.8128 | lr 1.00e-04 | 33.0s *best*
INFO    | epoch 3/5 | train loss 0.5797 acc 0.7980 | val loss 0.4930 acc 0.8330 | lr 1.00e-04 | 33

## 7. Đánh giá test cho từng run mới (bỏ qua run cũ kéo về từ GitHub)

In [7]:
import glob, os, subprocess, time
cutoff = time.time() - 24 * 3600  # chỉ đánh giá run mới trong 24 giờ qua
for ckpt in sorted(glob.glob("results/runs/*/resnet18-*/checkpoints/best.pt")) + sorted(glob.glob("results/runs/*/vgg11-*/checkpoints/best.pt")) + sorted(glob.glob("results/runs/*/densenet121-*/checkpoints/best.pt")):
    if os.path.getmtime(ckpt) < cutoff:
        print("bo qua run cu:", ckpt)
        continue
    print("=", ckpt)
    subprocess.run(["python", "-m", "src.evaluation", "--checkpoint", ckpt], check=True)

= results/runs/20261030-154216/resnet18-lr0.001-bs32-0/checkpoints/best.pt
= results/runs/20261030-154307/resnet18-lr0.0001-bs32-0/checkpoints/best.pt
= results/runs/20261030-154307/resnet18-lr0.0001-bs64-1/checkpoints/best.pt
= results/runs/20261030-154307/resnet18-lr0.0005-bs32-2/checkpoints/best.pt
= results/runs/20261030-154307/resnet18-lr0.0005-bs64-3/checkpoints/best.pt
= results/runs/20261030-154307/resnet18-lr0.001-bs32-4/checkpoints/best.pt
= results/runs/20261030-154307/resnet18-lr0.001-bs64-5/checkpoints/best.pt
= results/runs/20261030-154307/resnet18-lr0.003-bs32-6/checkpoints/best.pt
= results/runs/20261030-154307/resnet18-lr0.003-bs64-7/checkpoints/best.pt
= results/runs/20261030-154307/resnet18-lr0.005-bs32-8/checkpoints/best.pt
= results/runs/20261030-154307/resnet18-lr0.005-bs64-9/checkpoints/best.pt
= results/runs/20261030-154307/vgg11-lr0.0001-bs32-10/checkpoints/best.pt
= results/runs/20261030-154307/vgg11-lr0.0001-bs64-11/checkpoints/best.pt
= results/runs/20261030

## 8. Nén kết quả để tải về

In [8]:
import glob, time
new_runs = [d for d in glob.glob("results/runs/*") if (time.time() - os.path.getmtime(d)) < 24 * 3600]
print("run moi:", new_runs)
!tar -czf /content/sweep-results.tar.gz $new_runs
print("Tải file /content/sweep-results.tar.gz ở panel Files bên trái, giải nén vào results/runs/ trong máy bạn.")

run moi: ['results/runs/20261030-154216', 'results/runs/20261029-212336', 'results/runs/20261025-150847', 'results/runs/training_runs.md', 'results/runs/20261030-154307', 'results/runs/20261029-220833', 'results/runs/20261030-023250', 'results/runs/20261030-043458', 'results/runs/20261026-115633', 'results/runs/20261026-121817', 'results/runs/20261026-120218']
tar: [results/runs/20261030-154216,: Cannot stat: No such file or directory
tar: results/runs/20261029-212336,: Cannot stat: No such file or directory
tar: results/runs/20261025-150847,: Cannot stat: No such file or directory
tar: results/runs/training_runs.md,: Cannot stat: No such file or directory
tar: results/runs/20261030-154307,: Cannot stat: No such file or directory
tar: results/runs/20261029-220833,: Cannot stat: No such file or directory
tar: results/runs/20261030-023250,: Cannot stat: No such file or directory
tar: results/runs/20261030-043458,: Cannot stat: No such file or directory
tar: results/runs/20261026-115633,: